In [ ]:
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from transformers import CLIPProcessor, CLIPModel
import umap
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import pandas as pd
from PIL import Image

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
dataset_path = "../../dataset/CIFAKE/train/FAKE/"

In [ ]:
import os
def create_dataset(root, d_type):
    rf_dir = os.listdir(f"{root}{d_type}/") # Real or Fake
    allfiles = []
    for rf in rf_dir:
        class_dir = os.listdir(f"{root}{d_type}/{rf}/")
        for dir in class_dir:
            files = os.listdir(f"{root}{d_type}/{rf}/{dir}/")
            for f in files:
                filepath = f"{root}{d_type}/{rf}/{dir}/{f}"
                allfiles.append({"filepath":filepath, "class name": dir, "rf":rf})
    return allfiles

In [ ]:
root = "../../dataset/CIFAKE/"
d_type = "train"
allfiles = create_dataset(root, d_type)
df_train = pd.DataFrame(allfiles)
df_train.loc[df_train["rf"] == "REAL", "label"] = 0
df_train.loc[df_train["rf"] == "FAKE", "label"] = 1
df_train["label"] = df_train["label"].astype(int)

In [ ]:
class_list = sorted(df_train["class name"].unique())  # ソートして安定化
class_to_id = {name: i for i, name in enumerate(class_list)}
df_train["class_id"] = df_train["class name"].map(class_to_id)
id_to_class = {v: k for k, v in class_to_id.items()}

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [ ]:
class TransDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        filepath = self.dataframe.iloc[idx]["filepath"]
        image = Image.open(filepath).convert("RGB")
        label = self.dataframe.iloc[idx]["label"]
        class_id = self.dataframe.iloc[idx]["class_id"]
        
        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long), class_id

In [ ]:
dataset = TransDataset(df_test)

In [ ]:
images, labels, class_indices = [], [], []
for i in range(len(dataset)):
    img, label, class_id = dataset[i]
    images.append(img)
    labels.append(label)
    class_indices.append(class_id)

In [ ]:
image_embeddings = []
with torch.no_grad():
    for img in tqdm(images):
        inputs = processor(images=img, return_tensors="pt").to(device)
        outputs = model.get_image_features(**inputs)
        image_embeddings.append(outputs.cpu().numpy().flatten())

In [ ]:
image_embeddings = np.array(image_embeddings)
labels = np.array(labels)
class_indices = np.array(class_indices)

In [ ]:
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine")
embeddings_2d = reducer.fit_transform(image_embeddings)